# اخراجات کے دعوے کا تجزیہ

یہ نوٹ بک یہ ظاہر کرتی ہے کہ کس طرح ایجنٹس بنائے جائیں جو پلگ انز کا استعمال کرتے ہوئے مقامی رسید کی تصاویر سے سفری اخراجات کو پراسیس کرتے ہیں، اخراجات کا دعوے کا ای میل جنریٹ کرتے ہیں، اور پائی چارٹ کے ذریعے اخراجات کا ڈیٹا بصری شکل میں پیش کرتے ہیں۔ ایجنٹس کام کے سیاق و سباق کی بنیاد پر متحرک طور پر فنکشنز کا انتخاب کرتے ہیں۔

مراحل:
1. OCR ایجنٹ مقامی رسید کی تصویر کو پراسیس کرتا ہے اور سفری اخراجات کا ڈیٹا نکالتا ہے۔
2. ای میل ایجنٹ اخراجات کے دعوے کا ای میل تیار کرتا ہے۔

### سفر کے اخراجات کی منظر نامے کی مثال:
تصور کریں کہ آپ ایک ملازم ہیں جو کاروباری اجلاس کے لیے کسی دوسرے شہر جا رہے ہیں۔ آپ کی کمپنی کی پالیسی ہے کہ تمام معقول سفری اخراجات کی واپسی کی جائے۔ یہاں ممکنہ سفری اخراجات کی تفصیل ہے:
- نقل و حمل:
آپ کے گھر کے شہر سے منزل کے شہر کے لیے راؤنڈ ٹرپ کا ہوائی کرایہ۔
ہوائی اڈے تک اور ہوائی اڈے سے ٹیکسی یا رائیڈ ہیلنگ خدمات۔
منزل کے شہر میں مقامی نقل و حمل (جیسے کہ پبلک ٹرانزٹ، کرائے کی گاڑیاں، یا ٹیکسی)۔

- قیام:
تین راتوں کا قیام ایک درمیانے درجے کے کاروباری ہوٹل میں جو اجلاس کے مقام کے قریب ہو۔

- کھانے:
کمپنی کی پریم ڈایئم پالیسی کے مطابق ناشتہ، دوپہر کا کھانا، اور رات کا کھانا کے لیے روزانہ کھانے کا الاونس۔

- متفرق اخراجات:
ہوائی اڈے پر پارکنگ فیس۔
ہوٹل میں انٹرنیٹ کی سہولت کے چارجز۔
ٹوپ یا چھوٹے سروس چارجز۔

- دستاویزات:
آپ تمام رسیدیں (پروازیں، ٹیکسی، ہوٹل، کھانے، وغیرہ) اور ایک مکمل اخراجات کی رپورٹ واپسی کے لیے جمع کراتے ہیں۔


## ضروری لائبریریاں درآمد کریں

نوٹ بک کے لیے ضروری لائبریریاں اور ماڈیول درآمد کریں۔


In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import base64
from typing import Annotated, List

from pydantic import BaseModel, Field

from agent_framework import tool, AgentResponseUpdate, WorkflowBuilder
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

 ## خرچ کے ماڈلز کی تعریف کریں

انفرادی اخراجات کے لیے ایک Pydantic ماڈل بنائیں اور ایک ExpenseFormatter کلاس بنائیں جو صارف کی کوئری کو ساختہ اخراجات کے اعداد و شمار میں تبدیل کرے۔

ہر خرچ کو درج ذیل فارمیٹ میں ظاہر کیا جائے گا:
 `{'date': '07-Mar-2025', 'description': 'flight to destination', 'amount': 675.99, 'category': 'Transportation'}`


In [ ]:
class Expense(BaseModel):
    date: str = Field(..., description="Date of expense in dd-MMM-yyyy format")
    description: str = Field(..., description="Expense description")
    amount: float = Field(..., description="Expense amount")
    category: str = Field(..., description="Expense category (e.g., Transportation, Meals, Accommodation, Miscellaneous)")

class ExpenseFormatter(BaseModel):
    raw_query: str = Field(..., description="Raw query input containing expense details")
    
    def parse_expenses(self) -> List[Expense]:
        """
        Parses the raw query into a list of Expense objects.
        Expected format: "date|description|amount|category" separated by semicolons.
        """
        expense_list = []
        for expense_str in self.raw_query.split(";"):
            if expense_str.strip():
                parts = expense_str.strip().split("|")
                if len(parts) == 4:
                    date, description, amount, category = parts
                    try:
                        expense = Expense(
                            date=date.strip(),
                            description=description.strip(),
                            amount=float(amount.strip()),
                            category=category.strip()
                        )
                        expense_list.append(expense)
                    except ValueError as e:
                        print(f"[LOG] Parse Error: Invalid data in '{expense_str}': {e}")
        return expense_list

## اوزار کی تعریف - ای میل تیار کرنا

اخراجات کے دعوے جمع کرانے کے لیے ای میل تیار کرنے کے لیے ایک ٹول فنکشن بنائیں۔
- یہ ٹول مائیکروسافٹ ایجنٹ فریم ورک سے `@tool` ڈیکوریٹر استعمال کرتا ہے۔
- یہ اخراجات کی کل رقم کا حساب لگاتا ہے اور تفصیلات کو ای میل کے متن میں ترتیب دیتا ہے۔


In [ ]:
@tool(approval_mode="never_require")
def generate_expense_email(
    expense_data: Annotated[str, "Semicolon-separated expense entries in 'date|description|amount|category' format"]
) -> str:
    """Generate an email to submit an expense claim to the Finance Team."""
    formatter = ExpenseFormatter(raw_query=expense_data)
    expenses = formatter.parse_expenses()
    if not expenses:
        return "No valid expenses found to include in the email."
    total_amount = sum(e.amount for e in expenses)
    email_body = "Dear Finance Team,\n\n"
    email_body += "Please find below the details of my expense claim:\n\n"
    for e in expenses:
        email_body += f"- {e.date} | {e.description}: ${e.amount:.2f} ({e.category})\n"
    email_body += f"\nTotal Amount: ${total_amount:.2f}\n\n"
    email_body += "Receipts for all expenses are attached for your reference.\n\n"
    email_body += "Thank you,\n[Your Name]"
    return email_body

# رسید کی تصاویر سے سفری اخراجات نکالنے کا آلہ

رسید کی تصاویر سے سفری اخراجات نکالنے کے لیے ایک آلہ کی فنکشن تخلیق کریں۔
- یہ آلہ Microsoft Agent Framework کے `@tool` ڈیکوریٹر کا استعمال کرتا ہے۔
- یہ رسید کی تصویر کو پڑھتا ہے، اسے base64 میں انکوڈ کرتا ہے، اور ایجنٹ کے تجزیہ کے لیے data URI واپس کرتا ہے۔


In [ ]:
@tool(approval_mode="never_require")
def load_receipt_image(
    image_path: Annotated[str, "Path to the receipt image file"] = "receipt.jpg"
) -> str:
    """Load a receipt image and return its base64-encoded data URI for OCR extraction."""
    try:
        with open(image_path, "rb") as f:
            image_data = base64.b64encode(f.read()).decode("utf-8")
        return f"data:image/jpeg;base64,{image_data}"
    except Exception as e:
        error_msg = f"[LOG] Error loading image '{image_path}': {str(e)}"
        print(error_msg)
        return error_msg

## اخراجات کی پروسیسنگ

ایجنٹس کی تعریف کریں اور انہیں `WorkflowBuilder` کا استعمال کرتے ہوئے ایک تسلسلی ورک فلو میں جوڑیں۔  
- OCR ایجنٹ رسید کی تصویر سے ساختہ شدہ خرچ کا ڈیٹا `load_receipt_image` ٹول کے ذریعے نکالتا ہے۔  
- ای میل ایجنٹ نکالی گئی معلومات لے کر `generate_expense_email` ٹول کے ذریعے ایک پروفیشنل خرچ کا دعویٰ ای میل تیار کرتا ہے۔  
- `WorkflowBuilder` میں `add_edge` کے ذریعے ایک تسلسلی پائپ لائن بنائیں: OCR ایجنٹ → ای میل ایجنٹ۔


In [ ]:
ocr_agent = await provider.create_agent(
    tools=[load_receipt_image],
    name="OCRAgent",
    instructions=(
        "You are an expert OCR assistant specialized in extracting structured data from receipt images. "
        "Use the 'load_receipt_image' tool to load the receipt image, then analyze it and extract "
        "travel-related expense details in the format: 'date|description|amount|category' separated by semicolons. "
        "Follow these rules: "
        "- Date: Convert dates (e.g., '4/4/22') to 'dd-MMM-yyyy' (e.g., '04-Apr-2022'). "
        "- Description: Extract item names. "
        "- Amount: Use numeric values (e.g., '4.50' from '$4.50'). "
        "- Category: Infer from context (e.g., 'Meals' for food, 'Transportation' for travel, "
        "'Accommodation' for lodging, 'Miscellaneous' otherwise). "
        "Ignore totals, subtotals, or service charges unless they are itemized expenses. "
        "If no expenses are found, return 'No expenses detected'. "
        "Return only the structured data, no additional text."
    ),
)

email_agent = await provider.create_agent(
    name="EmailAgent",
    instructions=(
        "You are an expense claim email generator. Take the travel expense data from the previous agent "
        "(in 'date|description|amount|category' format separated by semicolons) and use the "
        "'generate_expense_email' tool to produce a professional expense claim email. "
        "Pass the semicolon-separated expense data directly to the tool."
    ),
)

## مین فنکشن

سیاق و سباق کا کام بنائیں اور اسے چلائیں تاکہ رسید کی تصویر کو پروسیس کیا جا سکے اور خرچ کے دعوے کا ای میل تیار کیا جا سکے۔


In [ ]:
workflow = WorkflowBuilder(start_executor=ocr_agent) \
    .add_edge(ocr_agent, email_agent) \
    .build()

prompt = (
    "Please extract the raw text from the receipt image at 'receipt.jpg', "
    "focusing on travel expenses like dates, descriptions, amounts, and categories "
    "(e.g., Transportation, Accommodation, Meals, Miscellaneous). "
    "Then generate a professional expense claim email."
)

last_author = None
events = workflow.run(
    prompt,
    stream=True,
)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"# Agent - {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ذمہ داری سے معذرت**:
یہ دستاویز AI ترجمہ سروس [Co-op Translator](https://github.com/Azure/co-op-translator) کے ذریعے ترجمہ کی گئی ہے۔ اگرچہ ہم درستگی کے لیے کوشش کرتے ہیں، براہ کرم آگاہ رہیں کہ خودکار تراجم میں غلطیاں یا کمی بیشی ہو سکتی ہے۔ اصل دستاویز اپنی مادری زبان میں معتبر ماخذ تصور کی جانی چاہیے۔ اہم معلومات کے لیے پیشہ ورانہ انسانی ترجمہ تجویز کیا جاتا ہے۔ اس ترجمہ کے استعمال سے ہونے والی کسی بھی غلط فہمی یا تشریح کی ذمہ داری ہم پر نہیں ہوگی۔
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
